# Mall Dataset Motion + Density Check
This notebook demonstrates the implemented motion analysis and density analysis logic using the `mall_dataset` files. It loads the mall frames, inspects the available ground-truth `.mat` data, and applies both motion and density workflows to the same dataset.

In [1]:
# Setup imports and repository paths
import inspect
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

from src.motion_analysis import process, FramePairLoader
from src.density_estimation import (
    DEFAULT_SHANGHAITECH_ROOT,
    compute_density_map,
    draw_density_debug,
    get_head_points,
    process_frame_density,
)

print('Repo root:', repo_root)
print('Motion analysis process function loaded:', process.__name__)
print('Density analysis function loaded:', process_frame_density.__name__)
print('Density map builder loaded:', compute_density_map.__name__)

Repo root: c:\Users\Lokesh Kumar\OneDrive\Desktop\HS202_G18
Motion analysis process function loaded: process
Density analysis function loaded: process_frame_density
Density map builder loaded: compute_density_map


In [2]:
# Inspect implemented functions and module summaries
print('--- Motion Analysis ---')
print(inspect.getdoc(process).split('\n')[0:8])
print('\n--- Density Analysis ---')
print(inspect.getdoc(process_frame_density).split('\n')[0:8])

--- Motion Analysis ---
['Optimized processing loop for frame sequences.', '', 'Computes optical flow, magnitude, zone speeds, and generates visualization panels.', 'Saves 2×2 combined panel (original, heatmap, flow arrows, zone overlay) to output directory.', '', 'Args:', '    source: Path to image directory', '    display: If True, display results in a window (slower)']

--- Density Analysis ---
['Process one frame and return zone density output payload.']


In [3]:
# Dataset paths for mall dataset
mall_root = repo_root / 'mall_dataset'
frames_dir = mall_root / 'frames'
gt_path = mall_root / 'mall_gt.mat'
feat_path = mall_root / 'mall_feat.mat'
roi_path = mall_root / 'perspective_roi.mat'

print('Frames directory exists:', frames_dir.exists())
print('Ground truth file exists:', gt_path.exists())
print('Feature file exists:', feat_path.exists())
print('ROI file exists:', roi_path.exists())

frame_files = sorted(frames_dir.glob('*'))[:5]
print('Sample frames:', len(frame_files), [p.name for p in frame_files[:5]])

Frames directory exists: True
Ground truth file exists: True
Feature file exists: True
ROI file exists: True
Sample frames: 5 ['seq_000001.jpg', 'seq_000002.jpg', 'seq_000003.jpg', 'seq_000004.jpg', 'seq_000005.jpg']


In [4]:
# Inspect mall_gt.mat contents and prepare point extraction for mall dataset
import scipy.io
from typing import Any

def load_mat_file(path: Path) -> dict:
    return scipy.io.loadmat(str(path))

mall_gt = load_mat_file(gt_path)
mat_keys = [k for k in mall_gt.keys() if not k.startswith('__')]
print('mall_gt.mat keys:', mat_keys)

# Mall dataset specific extraction
point_sets = []
if 'frame' in mall_gt:
    frame_data = mall_gt['frame']
    print('frame_data type:', type(frame_data), 'shape:', getattr(frame_data, 'shape', None))
    print('frame_data dtype:', getattr(frame_data, 'dtype', None))
    
    # Handle object array case: frame_data[0] contains the actual array of structs
    if frame_data.dtype == object and frame_data.size > 0:
        actual_frames = frame_data[0]  # Extract the inner array (2000 structs)
        print('actual_frames type:', type(actual_frames), 'shape:', getattr(actual_frames, 'shape', None))
        print('actual_frames dtype:', getattr(actual_frames, 'dtype', None))
        
        if hasattr(actual_frames, 'dtype') and actual_frames.dtype.names is not None:
            print('actual_frames is structured array with fields:', actual_frames.dtype.names)
            # Mall dataset: frame is array of structs, each with 'loc' field containing Nx2 points
            for i in range(len(actual_frames)):
                frame_struct = actual_frames[i]
                if 'loc' in frame_struct.dtype.names:
                    loc = frame_struct['loc']
                    if loc.size > 0 and loc.ndim == 2:
                        # loc should be Nx2 array of points
                        points = loc.astype(float)
                        point_sets.append((f'frame_{i}', points))
                        print(f'Frame {i}: extracted {points.shape[0]} points')
                    else:
                        print(f'Frame {i}: loc is empty or not 2D')
            
            print('Extracted point sets from mall_gt.mat:', len(point_sets))
            for key, points in point_sets[:3]:
                print(f'  {key}: {points.shape} points')
        else:
            print('actual_frames is not a structured array')
    elif hasattr(frame_data, 'dtype') and frame_data.dtype.names is not None:
        print('frame_data is structured array with fields:', frame_data.dtype.names)
        # Mall dataset: frame is array of structs, each with 'loc' field containing Nx2 points
        for i in range(len(frame_data)):
            frame_struct = frame_data[i]
            if 'loc' in frame_struct.dtype.names:
                loc = frame_struct['loc']
                if loc.size > 0 and loc.ndim == 2:
                    # loc should be Nx2 array of points
                    points = loc.astype(float)
                    point_sets.append((f'frame_{i}', points))
                    print(f'Frame {i}: extracted {points.shape[0]} points')
                else:
                    print(f'Frame {i}: loc is empty or not 2D')
        
        print('Extracted point sets from mall_gt.mat:', len(point_sets))
        for key, points in point_sets[:3]:
            print(f'  {key}: {points.shape} points')
    else:
        print('frame_data is not a structured array or object array')

# Fallback to generic extraction if specific failed
if not point_sets:
    def extract_point_arrays(obj: Any):
        """Recursively search for Nx2 point arrays in a loaded .mat object."""
        if isinstance(obj, np.ndarray):
            if obj.dtype == object:
                for cell in obj.flat:
                    yield from extract_point_arrays(cell)
                return
            if obj.ndim == 2 and obj.shape[1] == 2:
                yield obj.astype(float)
                return
            if obj.ndim == 2 and obj.shape[0] == 2:
                yield obj.T.astype(float)
                return
            for item in obj.flat:
                yield from extract_point_arrays(item)
            return
        if isinstance(obj, dict):
            for value in obj.values():
                yield from extract_point_arrays(value)
            return
        if isinstance(obj, (list, tuple)):
            for value in obj:
                yield from extract_point_arrays(value)

    for key in mat_keys:
        for points in extract_point_arrays(mall_gt[key]):
            if points.size > 0:
                point_sets.append((key, points))

print('Total found point set count:', len(point_sets))
for key, points in point_sets[:5]:
    print(' key=', key, 'points.shape=', points.shape)

# Keep one point set for demonstration
if point_sets:
    gt_key, demo_points = point_sets[0]
    print('Using first point set from key:', gt_key, 'with', demo_points.shape[0], 'points')
else:
    demo_points = np.empty((0, 2), dtype=float)
    print('No point sets found in mall_gt.mat; density demo will use an empty point set.')

mall_gt.mat keys: ['frame', 'count']
frame_data type: <class 'numpy.ndarray'> shape: (1, 2000)
frame_data dtype: object
actual_frames type: <class 'numpy.ndarray'> shape: (2000,)
actual_frames dtype: object
actual_frames is not a structured array
Total found point set count: 0
No point sets found in mall_gt.mat; density demo will use an empty point set.


In [5]:
# Build a grid for density analysis based on the first frame size
sample_frame = cv2.imread(str(frame_files[0]))
h, w = sample_frame.shape[:2]
rows, cols = 4, 4
cell_h = h // rows
cell_w = w // cols

mall_grid = {
    f'z{r*cols + c}': (
        c * cell_w,
        r * cell_h,
        (c + 1) * cell_w if c < cols - 1 else w,
        (r + 1) * cell_h if r < rows - 1 else h,
    )
    for r in range(rows)
    for c in range(cols)
}

print('Generated grid zones:', len(mall_grid))
print(list(mall_grid.items())[:3])

Generated grid zones: 16
[('z0', (0, 0, 160, 120)), ('z1', (160, 0, 320, 120)), ('z2', (320, 0, 480, 120))]


In [ ]:
# Density analysis for a sample mall frame using the first point set
frame_path = frame_files[0]
frame = cv2.imread(str(frame_path))

# If mall_gt contains a matching frame annotation table, it will be used.
# Otherwise this will still show the density pipeline applied to the first point set.
if demo_points.size:
    frame_id = frame_path.stem
    result = compute_density_map(
        points=[(float(x), float(y)) for x, y in demo_points],
        grid=mall_grid,
        frame_id=frame_id,
        normalize=False,
        normalization_mode='area',
    )
    print('Density result keys:', result.keys())
    print('Zone counts sample:', list(result['zone_density'].items())[:8])
else:
    print('Skipping density processing because no mall annotations were extracted.')

# Draw the debug overlay using extracted points if available
if demo_points.size:
    debug_frame = draw_density_debug(
        image=frame,
        grid=mall_grid,
        points=[(float(x), float(y)) for x, y in demo_points[:200]],
        zone_density=result['zone_density'],
        show_zone_counts=True,
    )
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(debug_frame, cv2.COLOR_BGR2RGB))
    plt.title('Mall Density Debug Overlay')
    plt.axis('off')
    plt.show()

In [ ]:
# Heatmap visualization similar to density_experiments_full.ipynb
if demo_points.size > 0:
    h, w = frame.shape[:2]
    density = np.zeros((h, w), dtype=np.float32)

    for x, y in demo_points:
        xi = int(round(x))
        yi = int(round(y))
        if 0 <= xi < w and 0 <= yi < h:
            density[yi, xi] += 1.0

    heat = cv2.GaussianBlur(density, (0, 0), sigmaX=15, sigmaY=15)
    heat_norm = cv2.normalize(heat, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    heat_color = cv2.applyColorMap(heat_norm, cv2.COLORMAP_JET)

    alpha = 0.55
    overlay_bgr = cv2.addWeighted(frame, 1 - alpha, heat_color, alpha, 0)

    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(heat_norm, cmap="jet")
    plt.title(f"Raw Heatmap: {frame_path.name}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB))
    plt.title(f"Heatmap Overlay: {frame_path.name}")
    plt.axis("off")
    plt.show()
else:
    print('No points available for heatmap generation.')

In [ ]:
# Motion analysis summary using the mall frames
motion_stats = process(
    str(frames_dir),
    display=False,
    save_frames=False,
    start_idx=0,
    end_idx=20,
    skip_frames=1,
)
print('Motion analysis stats:', motion_stats)

In [ ]:
# Frame-by-frame motion visualization similar to save_motion_frames.py
print('Generating frame-by-frame motion visualizations...')
motion_viz_stats = process(
    str(frames_dir),
    display=False,  # Set to True to show windows, False to run headless
    save_frames=True,  # Save visualization frames as PNG
    start_idx=0,
    end_idx=10,  # Process first 10 frames for demo
    skip_frames=1,
)
print('Motion visualization complete. Check output_frames/ for saved images.')
print('Stats:', motion_viz_stats)

## Summary
This notebook demonstrates both:

- **Motion analysis**: optical flow, magnitude, zone speed maps, and processing of mall frames with frame-by-frame visualization saved to `output_frames/`.
- **Density analysis**: extraction of mall ground truth points from `.mat`, grid-based density counting, debug overlay, and heatmap generation similar to `density_experiments_full.ipynb`.

The point extraction has been updated to properly handle the mall dataset's structured array format where points are stored in the 'loc' field of each frame struct.

If the mall dataset annotations are not in the expected structure, the notebook will still show how the density functions are used and print the `.mat` contents for further adaptation.